In [ ]:

# =====================================================
# NEXUS - PASO 1: CARGAR LOS ARCHIVOS CSV
# =====================================================

from google.colab import files
import pandas as pd
import io

print("Selecciona los 10 archivos CSV de NEXUS.")

archivos_subidos = files.upload()

# Leer los archivos y guardarlos en un diccionario
datos = {}

for nombre, contenido in archivos_subidos.items():
    df = pd.read_csv(io.BytesIO(contenido), sep=None, engine="python")
    datos[nombre] = df

    print(f"\nArchivo: {nombre}")
    print(f"Filas: {df.shape[0]}")
    print(f"Columnas: {df.shape[1]}")
    print("Nombres de columnas:")
    print(df.columns.tolist())

print("\nCarga finalizada.")
print("Total de archivos cargados:", len(datos))

Selecciona los 10 archivos CSV de NEXUS.


Saving dim_contenido_202610131626.csv to dim_contenido_202610131626.csv
Saving dim_dispositivo_202610131626.csv to dim_dispositivo_202610131626.csv
Saving dim_tiempo_202610131626.csv to dim_tiempo_202610131626.csv
Saving dim_tipo_interaccion_202610131626.csv to dim_tipo_interaccion_202610131626.csv
Saving dim_ubicacion_202610131626.csv to dim_ubicacion_202610131626.csv
Saving dim_usuarios_202610131626.csv to dim_usuarios_202610131626.csv
Saving fact_creacion_contenido_202610131626.csv to fact_creacion_contenido_202610131626.csv
Saving fact_impresiones_anuncios_202610131626.csv to fact_impresiones_anuncios_202610131626.csv
Saving fact_interacciones_usuario_202610131626.csv to fact_interacciones_usuario_202610131626.csv
Saving fact_sesiones_uso_202610131626.csv to fact_sesiones_uso_202610131626.csv

Archivo: dim_contenido_202610131626.csv
Filas: 10000
Columnas: 5
Nombres de columnas:
['contenido_sk', 'titulo', 'tipo_formato', 'longitud_rango', 'categoria_tema']

Archivo: dim_dispositivo_

In [ ]:

# =====================================================
# NEXUS - PASO 2: PREPARAR LOS DATOS
# Predicción 1: Nivel de riesgo de usuarios
# =====================================================

import pandas as pd

# Buscar automáticamente el archivo de usuarios
archivo_usuarios = next(
    nombre for nombre in datos
    if nombre.startswith("dim_usuarios")
)

usuarios = datos[archivo_usuarios].copy()

# Mostrar las primeras filas
print("Primeros registros de usuarios:")
display(usuarios.head())

# Revisar las columnas disponibles
print("\nColumnas disponibles:")
print(usuarios.columns.tolist())

# Revisar la distribución de los niveles de riesgo
print("\nDistribución de niveles de riesgo:")
print(usuarios["nivel_riesgo"].value_counts().sort_index())

# Revisar valores faltantes
print("\nValores faltantes por columna:")
print(usuarios.isnull().sum())

# Revisar el tamaño del conjunto de datos
print("\nCantidad de usuarios:", len(usuarios))


Primeros registros de usuarios:


,user_sk,id_operacional,nombre_usuario,genero,rango_edad,estado_cuenta,nivel_riesgo,categoria_riesgo,promedio_reacciones,porcentaje_usuarios,reportes_recibidos,acciones_bloqueadas,fecha_registro
0,501,NX_USR_00001,Mateo Quispe #1,Masculino,36-50,Activa,1,Muy Bajo,10.78,45.37,0,0,2026-09-16
1,502,NX_USR_00002,Alejandro Pérez #2,Femenino,15-20,Activa,1,Muy Bajo,26.92,45.37,0,0,2026-09-12
2,503,NX_USR_00003,Vania Sánchez #3,Masculino,50+,Activa,1,Muy Bajo,12.52,45.37,0,0,2026-09-20
3,504,NX_USR_00004,Vania Eccoña #4,Masculino,26-35,Activa,2,Bajo,22.06,24.80,1,0,2026-09-05
4,505,NX_USR_00005,Fernanda Díaz #5,Masculino,26-35,Activa,2,Bajo,13.56,24.80,1,0,2026-09-02



Columnas disponibles:
['user_sk', 'id_operacional', 'nombre_usuario', 'genero', 'rango_edad', 'estado_cuenta', 'nivel_riesgo', 'categoria_riesgo', 'promedio_reacciones', 'porcentaje_usuarios', 'reportes_recibidos', 'acciones_bloqueadas', 'fecha_registro']

Distribución de niveles de riesgo:
nivel_riesgo
1    4537
2    2480
3    1521
4     977
5     485
Name: count, dtype: int64

Valores faltantes por columna:
user_sk                0
id_operacional         0
nombre_usuario         0
genero                 0
rango_edad             0
estado_cuenta          0
nivel_riesgo           0
categoria_riesgo       0
promedio_reacciones    0
porcentaje_usuarios    0
reportes_recibidos     0
acciones_bloqueadas    0
fecha_registro         0
dtype: int64

Cantidad de usuarios: 10000


In [ ]:

# =========================================================
# NEXUS - PREDICCION 1
# Clasificacion del nivel de riesgo de los usuarios
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

# 1. Copiar los datos de usuarios
df = usuarios.copy()

# 2. Definir la variable que queremos predecir
y = df["nivel_riesgo"].astype(int)

# 3. Seleccionar variables disponibles
# Excluimos identificadores, nombres y la categoria
# que deriva directamente del nivel de riesgo.
columnas_numericas = [
    "promedio_reacciones",
    "porcentaje_usuarios",
    "reportes_recibidos",
    "acciones_bloqueadas"
]

columnas_categoricas = [
    "genero",
    "rango_edad",
    "estado_cuenta"
]

columnas_numericas = [
    c for c in columnas_numericas if c in df.columns
]
columnas_categoricas = [
    c for c in columnas_categoricas if c in df.columns
]

X = df[columnas_numericas + columnas_categoricas].copy()

# 4. Separar entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Preparar las variables
preparacion_numerica = Pipeline([
    ("imputacion", SimpleImputer(strategy="median"))
])

preparacion_categorica = Pipeline([
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("codificacion", OneHotEncoder(handle_unknown="ignore"))
])

preparacion = ColumnTransformer([
    ("numericas", preparacion_numerica, columnas_numericas),
    ("categoricas", preparacion_categorica, columnas_categoricas)
])

# 6. Entrenar Random Forest
modelo_riesgo = Pipeline([
    ("preparacion", preparacion),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

modelo_riesgo.fit(X_train, y_train)

# 7. Evaluar el modelo con datos que no vio al entrenar
predicciones_test = modelo_riesgo.predict(X_test)

print("PRECISION GLOBAL (Accuracy):",
      round(accuracy_score(y_test, predicciones_test), 4))

print("\nINFORME DE CLASIFICACION:")
print(classification_report(
    y_test,
    predicciones_test,
    zero_division=0
))

print("\nMATRIZ DE CONFUSION:")
print(confusion_matrix(
    y_test,
    predicciones_test,
    labels=[1, 2, 3, 4, 5]
))


PRECISION GLOBAL (Accuracy): 1.0

INFORME DE CLASIFICACION:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00       908
           2       1.00      1.00      1.00       496
           3       1.00      1.00      1.00       304
           4       1.00      1.00      1.00       195
           5       1.00      1.00      1.00        97

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000


MATRIZ DE CONFUSION:
[[908   0   0   0   0]
 [  0 496   0   0   0]
 [  0   0 304   0   0]
 [  0   0   0 195   0]
 [  0   0   0   0  97]]


In [ ]:

# =====================================================
# NEXUS - PASO 5: COMPROBAR LAS VARIABLES DE RIESGO
# =====================================================

# Revisar si las variables numéricas cambian según el riesgo
print("PROMEDIOS POR NIVEL DE RIESGO")

display(
    usuarios.groupby("nivel_riesgo")[
        [
            "promedio_reacciones",
            "porcentaje_usuarios",
            "reportes_recibidos",
            "acciones_bloqueadas"
        ]
    ].mean().round(2)
)

# Revisar la relación entre las variables numéricas
# y el nivel de riesgo
print("\nCORRELACIONES CON EL NIVEL DE RIESGO")

display(
    usuarios[
        [
            "nivel_riesgo",
            "promedio_reacciones",
            "porcentaje_usuarios",
            "reportes_recibidos",
            "acciones_bloqueadas"
        ]
    ].corr(numeric_only=True).round(3)
)


PROMEDIOS POR NIVEL DE RIESGO


,promedio_reacciones,porcentaje_usuarios,reportes_recibidos,acciones_bloqueadas
nivel_riesgo,,,,
1,20.29,45.37,0.00,0.00
2,20.68,24.80,0.51,0.00
3,8.18,15.21,1.99,0.50
4,4.55,9.77,5.49,2.97
5,1.88,4.85,18.53,11.04



CORRELACIONES CON EL NIVEL DE RIESGO


,nivel_riesgo,promedio_reacciones,porcentaje_usuarios,reportes_recibidos,acciones_bloqueadas
nivel_riesgo,1.000,-0.704,-0.954,0.754,0.697
promedio_reacciones,-0.704,1.000,0.624,-0.532,-0.491
porcentaje_usuarios,-0.954,0.624,1.000,-0.597,-0.525
reportes_recibidos,0.754,-0.532,-0.597,1.000,0.838
acciones_bloqueadas,0.697,-0.491,-0.525,0.838,1.000


In [ ]:

# =========================================================
# NEXUS - VALIDACION ALTERNATIVA DEL MODELO DE RIESGO
# Corregido: usar solo las columnas seleccionadas
# =========================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Preparar datos
df_riesgo = usuarios.copy()

columnas_numericas = ["promedio_reacciones"]
columnas_categoricas = ["genero", "rango_edad", "estado_cuenta"]

X = df_riesgo[columnas_numericas + columnas_categoricas].copy()
y = df_riesgo["nivel_riesgo"].astype(int)

# 2. Separar entrenamiento y prueba
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Preparar variables
preparacion2 = ColumnTransformer([
    (
        "numericas",
        SimpleImputer(strategy="median"),
        columnas_numericas
    ),
    (
        "categoricas",
        Pipeline([
            ("imputacion", SimpleImputer(strategy="most_frequent")),
            ("codificacion", OneHotEncoder(handle_unknown="ignore"))
        ]),
        columnas_categoricas
    )
])

# 4. Crear un modelo nuevo con esta configuración
modelo_riesgo_alternativo = Pipeline([
    ("preparacion", preparacion2),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 5. Entrenar
modelo_riesgo_alternativo.fit(X_train2, y_train2)

# 6. Evaluar
pred2 = modelo_riesgo_alternativo.predict(X_test2)

print("VARIABLES UTILIZADAS:")
print(X.columns.tolist())

print("\nEXACTITUD DEL MODELO ALTERNATIVO:")
print(f"{accuracy_score(y_test2, pred2):.4f}")

print("\nINFORME DE CLASIFICACION:")
print(classification_report(y_test2, pred2, zero_division=0))


VARIABLES UTILIZADAS:
['promedio_reacciones', 'genero', 'rango_edad', 'estado_cuenta']

EXACTITUD DEL MODELO ALTERNATIVO:
0.5730

INFORME DE CLASIFICACION:
              precision    recall  f1-score   support

           1       0.62      0.57      0.59       908
           2       0.36      0.38      0.37       496
           3       0.63      0.70      0.66       304
           4       0.76      0.75      0.75       195
           5       0.79      0.84      0.81        97

    accuracy                           0.57      2000
   macro avg       0.63      0.65      0.64      2000
weighted avg       0.58      0.57      0.57      2000



In [ ]:

# =========================================================
# NEXUS - PREDICCION 1
# Generar predicciones para los 10.000 usuarios
# =========================================================

# Preparar las variables con las que entrenamos el modelo
columnas_modelo = [
    "promedio_reacciones",
    "genero",
    "rango_edad",
    "estado_cuenta"
]

X_usuarios = usuarios[columnas_modelo].copy()

# Predecir el nivel de riesgo estimado
usuarios_predichos = usuarios.copy()

usuarios_predichos["riesgo_predicho"] = (
    modelo_riesgo_alternativo.predict(X_usuarios)
)

# Obtener la probabilidad estimada para cada nivel
probabilidades = modelo_riesgo_alternativo.predict_proba(X_usuarios)

# Guardar la probabilidad asociada al nivel predicho
usuarios_predichos["probabilidad_prediccion"] = (
    probabilidades.max(axis=1) * 100
).round(2)

# Mostrar las primeras 20 predicciones
columnas_resultado = [
    "id_operacional",
    "nombre_usuario",
    "nivel_riesgo",
    "riesgo_predicho",
    "probabilidad_prediccion"
]

display(usuarios_predichos[columnas_resultado].head(20))

print("Total de usuarios con predicción:", len(usuarios_predichos))

print("\nDistribución de riesgos predichos:")
print(
    usuarios_predichos["riesgo_predicho"]
    .value_counts()
    .sort_index()
)

# Exportar resultados a CSV
usuarios_predichos[columnas_resultado].to_csv(
    "predicciones_nexus_riesgo.csv",
    index=False
)

print("\nArchivo generado: predicciones_nexus_riesgo.csv")


,id_operacional,nombre_usuario,nivel_riesgo,riesgo_predicho,probabilidad_prediccion
0,NX_USR_00001,Mateo Quispe #1,1,1,88.00
1,NX_USR_00002,Alejandro Pérez #2,1,1,95.42
2,NX_USR_00003,Vania Sánchez #3,1,1,71.50
3,NX_USR_00004,Vania Eccoña #4,2,2,83.50
4,NX_USR_00005,Fernanda Díaz #5,2,2,72.50
5,NX_USR_00006,Fernanda Castillo #6,5,5,100.00
6,NX_USR_00007,Adriana Flores #7,2,2,87.50
7,NX_USR_00008,Sebastian Vergara #8,1,2,58.62
8,NX_USR_00009,Jose Morales #9,1,2,86.65
9,NX_USR_00010,Daniel Flores #10,1,1,92.22


Total de usuarios con predicción: 10000

Distribución de riesgos predichos:
riesgo_predicho
1    4277
2    2596
3    1669
4     961
5     497
Name: count, dtype: int64

Archivo generado: predicciones_nexus_riesgo.csv


In [ ]:

# =========================================================
# NEXUS - PREDICCION 2
# Deteccion de patrones de interaccion inusuales
# =========================================================

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Localizar el CSV de interacciones
archivo_interacciones = next(
    nombre for nombre in datos
    if nombre.startswith("fact_interacciones_usuario")
)

interacciones = datos[archivo_interacciones].copy()

# Variables de actividad disponibles
columnas = [
    "cantidad",
    "tiempo_en_pantalla_segundos"
]

X_anomalias = interacciones[columnas].copy()

# Convertir las variables a formato numérico
X_anomalias = X_anomalias.apply(
    pd.to_numeric, errors="coerce"
)

# Modelo de detección de anomalías
modelo_anomalias = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler()),
    ("modelo", IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42
    ))
])

# Entrenar y detectar registros atípicos
modelo_anomalias.fit(X_anomalias)

etiquetas = modelo_anomalias.predict(X_anomalias)
puntajes = modelo_anomalias.decision_function(X_anomalias)

resultado_anomalias = interacciones.copy()

resultado_anomalias["patron_detectado"] = [
    "Inusual" if etiqueta == -1 else "Habitual"
    for etiqueta in etiquetas
]

resultado_anomalias["puntaje_anomalia"] = puntajes.round(4)

# Mostrar resultados
print("DISTRIBUCION DE PATRONES")
print(resultado_anomalias["patron_detectado"].value_counts())

print("\nPRIMEROS 20 REGISTROS")
display(resultado_anomalias[
    [
        "user_sk",
        "cantidad",
        "tiempo_en_pantalla_segundos",
        "patron_detectado",
        "puntaje_anomalia"
    ]
].head(20))

# Exportar resultados
resultado_anomalias.to_csv(
    "predicciones_nexus_anomalias.csv",
    index=False
)

print("\nArchivo generado: predicciones_nexus_anomalias.csv")


DISTRIBUCION DE PATRONES
patron_detectado
Habitual    23823
Inusual      1177
Name: count, dtype: int64

PRIMEROS 20 REGISTROS


,user_sk,cantidad,tiempo_en_pantalla_segundos,patron_detectado,puntaje_anomalia
0,5212,2,44,Habitual,0.1058
1,6455,1,40,Habitual,0.0689
2,5158,1,60,Habitual,0.0815
3,3051,3,73,Habitual,0.0915
4,6709,3,116,Habitual,0.0099
5,6196,2,59,Habitual,0.1058
6,8463,1,17,Habitual,0.0129
7,7675,1,100,Habitual,0.0679
8,3500,2,52,Habitual,0.1058
9,5495,2,96,Habitual,0.0907



Archivo generado: predicciones_nexus_anomalias.csv


In [ ]:

# =========================================================
# NEXUS - PREDICCION 3
# Prediccion de abandono de sesion
# =========================================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Localizar y copiar la tabla de sesiones
archivo_sesiones = next(
    nombre for nombre in datos
    if nombre.startswith("fact_sesiones_uso")
)

sesiones = datos[archivo_sesiones].copy()

# 2. Definir la variable que queremos predecir
y = sesiones["abandono_sesion"].astype(int)

# 3. Seleccionar variables disponibles antes del resultado
columnas_numericas = [
    "duracion_sesion_segundos",
    "cantidad_publicaciones_vistas",
    "reinicios_feed"
]

columnas_categoricas = [
    "dispositivo_sk",
    "ubicacion_sk",
    "fecha_sk"
]

columnas_numericas = [
    c for c in columnas_numericas if c in sesiones.columns
]
columnas_categoricas = [
    c for c in columnas_categoricas if c in sesiones.columns
]

X = sesiones[columnas_numericas + columnas_categoricas].copy()

# Evitar que una duración final o el resultado de abandono
# se utilicen como si fueran información disponible al inicio.
# La duración se mantiene inicialmente para una evaluación
# exploratoria, pero luego debemos verificar cuándo se mide.

# 4. Separar entrenamiento y prueba
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Preparar variables
preparacion3 = ColumnTransformer([
    (
        "numericas",
        SimpleImputer(strategy="median"),
        columnas_numericas
    ),
    (
        "categoricas",
        Pipeline([
            ("imputacion", SimpleImputer(strategy="most_frequent")),
            ("codificacion", OneHotEncoder(handle_unknown="ignore"))
        ]),
        columnas_categoricas
    )
])

# 6. Crear y entrenar el modelo
modelo_abandono = Pipeline([
    ("preparacion", preparacion3),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

modelo_abandono.fit(X_train3, y_train3)

# 7. Evaluar el modelo
predicciones_abandono = modelo_abandono.predict(X_test3)

print("DISTRIBUCION REAL DE ABANDONO:")
print(y.value_counts().sort_index())

print("\nEXACTITUD DEL MODELO:")
print(round(
    accuracy_score(y_test3, predicciones_abandono), 4
))

print("\nINFORME DE CLASIFICACION:")
print(classification_report(
    y_test3,
    predicciones_abandono,
    zero_division=0
))

print("\nMATRIZ DE CONFUSION:")
print(confusion_matrix(
    y_test3,
    predicciones_abandono,
    labels=[0, 1]
))


DISTRIBUCION REAL DE ABANDONO:
abandono_sesion
0    11676
1     3324
Name: count, dtype: int64

EXACTITUD DEL MODELO:
0.8313

INFORME DE CLASIFICACION:
              precision    recall  f1-score   support

           0       0.86      0.94      0.90      2335
           1       0.67      0.47      0.55       665

    accuracy                           0.83      3000
   macro avg       0.77      0.70      0.72      3000
weighted avg       0.82      0.83      0.82      3000


MATRIZ DE CONFUSION:
[[2184  151]
 [ 355  310]]


In [ ]:

# =========================================================
# NEXUS - PREDICCION 3
# Generar predicciones para las 15.000 sesiones
# =========================================================

# Usar las mismas columnas del entrenamiento
X_sesiones = sesiones[
    columnas_numericas + columnas_categoricas
].copy()

# Predecir clase y probabilidad estimada
clases = modelo_abandono.predict(X_sesiones)
probabilidades = modelo_abandono.predict_proba(X_sesiones)

# Construir la tabla de resultados
predicciones_sesiones = sesiones.copy()

predicciones_sesiones["abandono_predicho"] = clases

# Probabilidad estimada de la clase 1 (abandono)
indice_clase_1 = list(
    modelo_abandono.named_steps["modelo"].classes_
).index(1)

predicciones_sesiones["probabilidad_abandono_pct"] = (
    probabilidades[:, indice_clase_1] * 100
).round(2)

# Mostrar una muestra
columnas_mostrar = [
    "sesion_sk",
    "user_sk",
    "abandono_sesion",
    "abandono_predicho",
    "probabilidad_abandono_pct"
]

display(predicciones_sesiones[columnas_mostrar].head(20))

print("Total de sesiones procesadas:", len(predicciones_sesiones))

print("\nDistribución de abandonos predichos:")
print(
    predicciones_sesiones["abandono_predicho"]
    .value_counts()
    .sort_index()
)

# Exportar el resultado
predicciones_sesiones[columnas_mostrar].to_csv(
    "predicciones_nexus_abandono.csv",
    index=False
)

print("\nArchivo generado: predicciones_nexus_abandono.csv")


,sesion_sk,user_sk,abandono_sesion,abandono_predicho,probabilidad_abandono_pct
0,2942,8227,0,0,3.0
1,2943,4575,0,0,3.0
2,2944,7877,1,1,62.0
3,2945,3310,0,0,1.5
4,2946,10185,1,1,81.5
5,2947,8562,1,1,64.5
6,2948,1290,0,0,4.0
7,2949,4113,1,1,68.5
8,2950,4945,1,1,90.5
9,2951,2965,0,0,13.5


Total de sesiones procesadas: 15000

Distribución de abandonos predichos:
abandono_predicho
0    11880
1     3120
Name: count, dtype: int64

Archivo generado: predicciones_nexus_abandono.csv


In [ ]:

# =========================================================
# NEXUS - PREDICCION 4
# Priorizacion heuristica de revision preventiva
# =========================================================

import pandas as pd

# Copiar los datos de usuarios
revision = usuarios.copy()

# Convertir las variables a valores numericos
revision["nivel_riesgo"] = pd.to_numeric(
    revision["nivel_riesgo"], errors="coerce"
)

revision["reportes_recibidos"] = pd.to_numeric(
    revision["reportes_recibidos"], errors="coerce"
)

revision["acciones_bloqueadas"] = pd.to_numeric(
    revision["acciones_bloqueadas"], errors="coerce"
)

# Normalizar variables a una escala de 0 a 1
def normalizar(columna):
    minimo = columna.min()
    maximo = columna.max()

    if pd.isna(minimo) or pd.isna(maximo) or maximo == minimo:
        return pd.Series(0.0, index=columna.index)

    return (columna - minimo) / (maximo - minimo)

revision["riesgo_normalizado"] = (
    (revision["nivel_riesgo"] - 1) / 4
)

revision["reportes_normalizados"] = normalizar(
    revision["reportes_recibidos"]
)

revision["bloqueos_normalizados"] = normalizar(
    revision["acciones_bloqueadas"]
)

# Puntaje heuristico con pesos definidos
revision["puntaje_revision"] = (
    0.50 * revision["riesgo_normalizado"] +
    0.30 * revision["reportes_normalizados"] +
    0.20 * revision["bloqueos_normalizados"]
) * 100

revision["puntaje_revision"] = (
    revision["puntaje_revision"].round(2)
)

# Clasificar prioridad de revisión
revision["prioridad_revision"] = pd.cut(
    revision["puntaje_revision"],
    bins=[-1, 35, 65, 100],
    labels=["Baja", "Media", "Alta"]
)

# Mostrar resultados
columnas_revision = [
    "id_operacional",
    "nivel_riesgo",
    "reportes_recibidos",
    "acciones_bloqueadas",
    "puntaje_revision",
    "prioridad_revision"
]

print("DISTRIBUCION DE PRIORIDAD")
print(revision["prioridad_revision"].value_counts())

print("\nPRIMEROS 20 USUARIOS")
display(revision[columnas_revision].head(20))

# Exportar resultados
revision[columnas_revision].to_csv(
    "predicciones_nexus_revision_preventiva.csv",
    index=False
)

print("\nArchivo generado: predicciones_nexus_revision_preventiva.csv")


DISTRIBUCION DE PRIORIDAD
prioridad_revision
Baja     8538
Media     991
Alta      471
Name: count, dtype: int64

PRIMEROS 20 USUARIOS


,id_operacional,nivel_riesgo,reportes_recibidos,acciones_bloqueadas,puntaje_revision,prioridad_revision
0,NX_USR_00001,1,0,0,0.00,Baja
1,NX_USR_00002,1,0,0,0.00,Baja
2,NX_USR_00003,1,0,0,0.00,Baja
3,NX_USR_00004,2,1,0,13.50,Baja
4,NX_USR_00005,2,1,0,13.50,Baja
5,NX_USR_00006,5,15,16,82.78,Alta
6,NX_USR_00007,2,1,0,13.50,Baja
7,NX_USR_00008,1,0,0,0.00,Baja
8,NX_USR_00009,1,0,0,0.00,Baja
9,NX_USR_00010,1,0,0,0.00,Baja



Archivo generado: predicciones_nexus_revision_preventiva.csv


In [ ]:

from google.colab import files

# Descargar predicciones de riesgo
files.download("predicciones_nexus_riesgo.csv")

# Descargar detección de anomalías
files.download("predicciones_nexus_anomalias.csv")

# Descargar predicciones de abandono
files.download("predicciones_nexus_abandono.csv")

# Descargar priorización preventiva
files.download("predicciones_nexus_revision_preventiva.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>